# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hisham-Walid/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook trains and audits the first learned model for the **CTR / engagement opportunity** lane. It ranks pages for human review; it does not prescribe automatic edits or claim causal uplift.

## 1. Method choice and why

The target is continuous future missed-click opportunity, and the operational question is “which pages should an editor inspect first?” I start with a **Random Forest regressor** on `log1p(target)`. A shallow, leaf-regularized forest can represent nonlinear interactions between volume, CTR, position, and history coverage without requiring a fragile linear relationship; the log transform reduces domination by a few very large opportunities.

Complexity still has to earn its place. I compare it with both a constant dummy floor and the frozen Week-4 rule structure on the exact same held-out clients. The learned model receives only the five pre-decision features established in ML-04. Identifiers, future outcomes, trend labels, and product flags are excluded. Random seeds and library versions are printed for reproducibility.

In [1]:
import json
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, ndcg_score
from sklearn.model_selection import GroupShuffleSplit

RANDOM_SEED = 42
FEATURE_START = '2026-03-01'
DECISION_DATE = '2026-03-20'
OUTCOME_START = '2026-03-21'
OUTCOME_END = '2026-03-31'
MIN_CLIENT_ROWS = 100
K = 20

repo_root = Path.cwd().resolve()
while not (repo_root / 'work' / 'notebooks').exists():
    if repo_root.parent == repo_root:
        raise FileNotFoundError('Run this notebook from somewhere inside the repository.')
    repo_root = repo_root.parent
output_dir = repo_root / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
local_file = os.getenv('FLYRANK_MARCH_PARQUET')
hf_token = os.getenv('HF_TOKEN')
if local_file and Path(local_file).is_file():
    escaped = Path(local_file).as_posix().replace("'", "''")
    FACT = f"read_parquet('{escaped}')"
    source_mode = 'authenticated local cache'
elif hf_token:
    escaped_token = hf_token.replace("'", "''")
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
    FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
    source_mode = 'authenticated Hugging Face stream'
else:
    raise RuntimeError('Set HF_TOKEN or FLYRANK_MARCH_PARQUET; never paste a token into this notebook.')

feature_query = f"""
WITH daily AS (
    SELECT report_date, client_hash_id, content_hash_id,
           gsc_impressions, gsc_clicks, gsc_sum_position
    FROM {FACT}
    WHERE gsc_data_available IS TRUE
), prior AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS prior_impressions,
           SUM(gsc_clicks) AS prior_clicks,
           SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS prior_avg_position,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS prior_active_days,
           100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS prior_ctr_pct
    FROM daily
    WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
), outcome AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS future_impressions,
           SUM(gsc_clicks) AS future_clicks,
           SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS future_avg_position,
           100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS future_ctr_pct
    FROM daily
    WHERE report_date BETWEEN DATE '{OUTCOME_START}' AND DATE '{OUTCOME_END}'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
)
SELECT p.*, o.* EXCLUDE (client_hash_id, content_hash_id)
FROM prior p
JOIN outcome o USING (client_hash_id, content_hash_id)
"""
model_frame = con.execute(feature_query).df()
client_counts = model_frame.groupby('client_hash_id').size()
eligible_clients = client_counts[client_counts >= MIN_CLIENT_ROWS].index
model_frame = model_frame.loc[model_frame['client_hash_id'].isin(eligible_clients)].copy()
model_frame = model_frame.sort_values(['client_hash_id', 'content_hash_id']).reset_index(drop=True)

feature_columns = [
    'prior_impressions', 'prior_clicks', 'prior_avg_position',
    'prior_active_days', 'prior_ctr_pct',
]
forbidden = {
    'client_hash_id', 'content_hash_id', 'future_impressions', 'future_clicks',
    'future_avg_position', 'future_ctr_pct', 'future_missed_clicks',
    'trend_pct', 'trend_direction', 'is_declining_label',
}
assert model_frame[['client_hash_id', 'content_hash_id']].duplicated().sum() == 0
assert len(feature_columns) == 5 and set(feature_columns).isdisjoint(forbidden)
assert model_frame[feature_columns].notna().all().all()
print(f'Warehouse ready via {source_mode}: {len(model_frame):,} pages across {model_frame.client_hash_id.nunique()} clients.')
print(f'Versions — scikit-learn {sklearn.__version__}; pandas {pd.__version__}; DuckDB {duckdb.__version__}.')

Warehouse ready via authenticated local cache: 86,574 pages across 22 clients.
Versions — scikit-learn 1.9.0; pandas 3.0.5; DuckDB 1.5.5.


## 2. Split design

The design has two protections. First, time runs forward: features end on March 20 and the target is observed on March 21–31. Second, `GroupShuffleSplit` holds out whole clients, so the same client cannot appear in training and evaluation. The seed is fixed at 42.

I keep clients with at least 100 eligible page rows so NDCG@20 is meaningful rather than dominated by one-item clients. Future position bands define the outcome proxy, but their expected CTR values are estimated from **training clients only** and then applied to held-out outcomes. Likewise, the Week-4 baseline's peer CTR is calibrated from training-window features only. No test-client information tunes either method.

Limitation: this is one mid-panel month with a coverage filter. It estimates generalization to other observed clients in this month, not seasonality, newly onboarded clients, or causal response to editing. June stays sealed.

In [2]:
position_bins = [-np.inf, 3, 10, 20, 50, np.inf]
position_labels = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
model_frame['future_position_band'] = pd.cut(
    model_frame['future_avg_position'], bins=position_bins, labels=position_labels
)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(model_frame, groups=model_frame['client_hash_id']))
train = model_frame.iloc[train_idx].copy()
test = model_frame.iloc[test_idx].copy()

train_peer = (
    train.groupby('future_position_band', observed=True)
    .agg(impressions=('future_impressions', 'sum'), clicks=('future_clicks', 'sum'))
)
train_peer['expected_future_ctr_pct'] = 100 * train_peer['clicks'] / train_peer['impressions']
for frame in (train, test):
    frame['expected_future_ctr_pct'] = (
        frame['future_position_band']
        .map(train_peer['expected_future_ctr_pct'])
        .astype(float)
    )
    frame['future_ctr_gap_pp'] = (
        frame['expected_future_ctr_pct'] - frame['future_ctr_pct']
    ).clip(lower=0)
    frame['future_missed_clicks'] = (
        frame['future_impressions'] * frame['future_ctr_gap_pp'] / 100
    )

assert set(train['client_hash_id']).isdisjoint(set(test['client_hash_id']))
assert (test['future_missed_clicks'] >= 0).all()
split_summary = pd.DataFrame({
    'split': ['train', 'held-out test'],
    'pages': [len(train), len(test)],
    'clients': [train['client_hash_id'].nunique(), test['client_hash_id'].nunique()],
    'positive_target_rate': [
        (train['future_missed_clicks'] > 0).mean(),
        (test['future_missed_clicks'] > 0).mean(),
    ],
})
display(split_summary.style.format({'positive_target_rate': '{:.1%}'}))
print(f'Feature window: {FEATURE_START} to {DECISION_DATE}; outcome: {OUTCOME_START} to {OUTCOME_END}.')
print('Client overlap: 0. June 2026 was not read.')

,split,pages,clients,positive_target_rate
0,train,76321,16,73.0%
1,held-out test,10253,6,62.6%


Feature window: 2026-03-01 to 2026-03-20; outcome: 2026-03-21 to 2026-03-31.
Client overlap: 0. June 2026 was not read.


## 3. Train and compare with the baseline

The baseline keeps ML-07's exact structure: at least 500 prior impressions, prior position 4–20, CTR below the weighted rate for the same coarse position bucket, and estimated missed clicks as the score. Only the two peer rates are recalibrated on training clients so the 20-day warehouse window is comparable and held-out clients stay untouched.

The primary metric is median client NDCG@20. Mean client NDCG@20 shows dispersion, mean client precision@20 shows whether the top queue contains any positive observed opportunity, and MAE keeps the score scale honest. A constant dummy provides the base floor. All methods use the same test pages, clients, target, and metrics.

In [3]:
for frame in (train, test):
    frame['prior_position_band'] = pd.cut(
        frame['prior_avg_position'], bins=[3, 10, 20], labels=['page_1', 'striking']
    )
baseline_train = train.loc[
    (train['prior_impressions'] >= 500) & train['prior_avg_position'].between(4, 20)
]
baseline_peer = (
    baseline_train.groupby('prior_position_band', observed=True)
    .agg(impressions=('prior_impressions', 'sum'), clicks=('prior_clicks', 'sum'))
)
baseline_peer['benchmark_ctr_pct'] = 100 * baseline_peer['clicks'] / baseline_peer['impressions']
test_baseline_benchmark = (
    test['prior_position_band'].map(baseline_peer['benchmark_ctr_pct']).astype(float)
)
baseline_eligible = (
    (test['prior_impressions'] >= 500) & test['prior_avg_position'].between(4, 20)
)
test['baseline_score'] = np.where(
    baseline_eligible,
    test['prior_impressions'] * (test_baseline_benchmark - test['prior_ctr_pct']).clip(lower=0) / 100,
    0.0,
)

dummy = DummyRegressor(strategy='mean')
dummy.fit(train[feature_columns], train['future_missed_clicks'])
test['dummy_score'] = dummy.predict(test[feature_columns])

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=20,
    max_features=0.8,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
model.fit(train[feature_columns], np.log1p(train['future_missed_clicks']))
test['model_score'] = np.expm1(model.predict(test[feature_columns])).clip(min=0)

def evaluate_scores(name, score_column):
    client_ndcg = []
    client_precision = []
    for _, group in test.groupby('client_hash_id'):
        client_ndcg.append(ndcg_score(
            [group['future_missed_clicks'].to_numpy()],
            [group[score_column].to_numpy()],
            k=K,
        ))
        top_k = group.nlargest(min(K, len(group)), score_column)
        client_precision.append((top_k['future_missed_clicks'] > 0).mean())
    return {
        'method': name,
        'median_client_ndcg_at_20': float(np.median(client_ndcg)),
        'mean_client_ndcg_at_20': float(np.mean(client_ndcg)),
        'mean_client_precision_at_20': float(np.mean(client_precision)),
        'mae_future_missed_clicks': float(mean_absolute_error(
            test['future_missed_clicks'], test[score_column]
        )),
    }

comparison = pd.DataFrame([
    evaluate_scores('Dummy mean', 'dummy_score'),
    evaluate_scores('Week-4 rule baseline', 'baseline_score'),
    evaluate_scores('Random Forest', 'model_score'),
])
test_base_rate = (test['future_missed_clicks'] > 0).mean()
comparison['held_out_positive_base_rate'] = test_base_rate
baseline_ndcg = comparison.loc[
    comparison['method'] == 'Week-4 rule baseline', 'median_client_ndcg_at_20'
].iloc[0]
model_ndcg = comparison.loc[
    comparison['method'] == 'Random Forest', 'median_client_ndcg_at_20'
].iloc[0]
relative_ndcg_gain = model_ndcg / baseline_ndcg - 1
display(comparison.style.format({
    'median_client_ndcg_at_20': '{:.3f}',
    'mean_client_ndcg_at_20': '{:.3f}',
    'mean_client_precision_at_20': '{:.1%}',
    'mae_future_missed_clicks': '{:.3f}',
    'held_out_positive_base_rate': '{:.1%}',
}))
print(f'Random Forest relative median NDCG@20 gain over the rule: {relative_ndcg_gain:.1%}.')
print('The model wins the primary NDCG metric; the rule retains higher precision@20, so the finding is mixed rather than a universal model win.')

,method,median_client_ndcg_at_20,mean_client_ndcg_at_20,mean_client_precision_at_20,mae_future_missed_clicks,held_out_positive_base_rate
0,Dummy mean,0.044,0.046,68.3%,1.237,62.6%
1,Week-4 rule baseline,0.690,0.573,91.7%,0.486,62.6%
2,Random Forest,0.797,0.693,85.0%,0.430,62.6%


Random Forest relative median NDCG@20 gain over the rule: 15.5%.
The model wins the primary NDCG metric; the rule retains higher precision@20, so the finding is mixed rather than a universal model win.


## 4. Errors and interpretation

The model's largest absolute errors concentrate in the highest prior-impression quartile. That is plausible because the continuous target multiplies a CTR gap by future volume, making rare high-volume misses expensive. The three concrete cases below are all under-predictions: one combines very high prior volume with almost no clicks, one sits just above the baseline's position boundary, and one has only four active feature-window days. These cases show why a review queue still needs measurement and coverage checks.

Permutation importance is computed on held-out client NDCG@20. The expected leaders are prior impressions, prior CTR, and prior position: volume sets the opportunity scale, CTR indicates under-capture, and position supplies the peer context. Correlated impressions and clicks can share importance, so the table is directional evidence about this fitted model—not a causal explanation.

In [4]:
test_group_lookup = test['client_hash_id']

def held_out_grouped_ndcg(estimator, X, y_log):
    predictions = np.expm1(estimator.predict(X)).clip(min=0)
    actual = np.expm1(np.asarray(y_log))
    scored = pd.DataFrame({
        'client': test_group_lookup.loc[X.index].to_numpy(),
        'actual': actual,
        'prediction': predictions,
    })
    values = [
        ndcg_score([group['actual'].to_numpy()], [group['prediction'].to_numpy()], k=K)
        for _, group in scored.groupby('client')
    ]
    return float(np.median(values))

permutation = permutation_importance(
    model,
    test[feature_columns],
    np.log1p(test['future_missed_clicks']),
    scoring=held_out_grouped_ndcg,
    n_repeats=5,
    random_state=RANDOM_SEED,
    n_jobs=1,
)
rationales = {
    'prior_impressions': 'Sets the scale of recoverable click opportunity.',
    'prior_clicks': 'Adds observed click evidence beyond the rounded CTR.',
    'prior_avg_position': 'Provides the search-position context for attainable CTR.',
    'prior_active_days': 'Distinguishes stable history from a thin observation window.',
    'prior_ctr_pct': 'Measures earlier click capture before the outcome begins.',
}
importance = pd.DataFrame({
    'feature': feature_columns,
    'median_ndcg_drop': permutation.importances_mean,
    'repeat_sd': permutation.importances_std,
})
importance['why_plausible'] = importance['feature'].map(rationales)
importance = importance.sort_values('median_ndcg_drop', ascending=False).reset_index(drop=True)

test_errors = test.copy()
test_errors['absolute_error'] = (
    test_errors['future_missed_clicks'] - test_errors['model_score']
).abs()
test_errors['error_direction'] = np.where(
    test_errors['model_score'] < test_errors['future_missed_clicks'],
    'under-prediction',
    'over-prediction',
)
test_errors['prior_impression_quartile'] = pd.qcut(
    test_errors['prior_impressions'],
    q=4,
    labels=['Q1 lowest', 'Q2', 'Q3', 'Q4 highest'],
)
error_by_volume = (
    test_errors.groupby('prior_impression_quartile', observed=True)
    .agg(
        n=('future_missed_clicks', 'size'),
        mean_actual=('future_missed_clicks', 'mean'),
        mae=('absolute_error', 'mean'),
        under_prediction_pct=('error_direction', lambda values: 100 * (values == 'under-prediction').mean()),
    )
    .reset_index()
)
worst_cases = test_errors.nlargest(3, 'absolute_error').copy()
worst_cases.insert(0, 'case', ['case_1', 'case_2', 'case_3'])
worst_case_columns = [
    'case', 'prior_impressions', 'prior_clicks', 'prior_avg_position',
    'prior_active_days', 'prior_ctr_pct', 'future_missed_clicks',
    'model_score', 'baseline_score', 'absolute_error', 'error_direction',
]
print('Held-out permutation importance (drop in median client NDCG@20):')
display(importance.style.format({'median_ndcg_drop': '{:.3f}', 'repeat_sd': '{:.3f}'}))
print('Error by prior-impression quartile:')
display(error_by_volume.style.format({
    'mean_actual': '{:.3f}', 'mae': '{:.3f}', 'under_prediction_pct': '{:.1f}',
}))
print('Three largest held-out errors (identifiers intentionally omitted):')
display(worst_cases[worst_case_columns].style.format({
    'prior_avg_position': '{:.2f}', 'prior_ctr_pct': '{:.3f}',
    'future_missed_clicks': '{:.2f}', 'model_score': '{:.2f}',
    'baseline_score': '{:.2f}', 'absolute_error': '{:.2f}',
}))

metrics = {
    'assignment': 'ML-08',
    'seed': RANDOM_SEED,
    'feature_window': [FEATURE_START, DECISION_DATE],
    'outcome_window': [OUTCOME_START, OUTCOME_END],
    'model_rows': int(len(model_frame)),
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'train_clients': int(train['client_hash_id'].nunique()),
    'test_clients': int(test['client_hash_id'].nunique()),
    'test_positive_base_rate': round(float(test_base_rate), 4),
    'relative_median_ndcg_gain_vs_baseline': round(float(relative_ndcg_gain), 4),
    'comparison': comparison.round(6).to_dict(orient='records'),
    'permutation_importance': importance.round(6).to_dict(orient='records'),
    'versions': {
        'scikit_learn': sklearn.__version__,
        'pandas': pd.__version__,
        'duckdb': duckdb.__version__,
    },
}
metrics_path = output_dir / 'ml08_model_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')
print('Wrote reproducible metrics receipt to work/outputs/ml08_model_metrics.json.')

Held-out permutation importance (drop in median client NDCG@20):


,feature,median_ndcg_drop,repeat_sd,why_plausible
0,prior_impressions,0.747,0.007,Sets the scale of recoverable click opportunity.
1,prior_ctr_pct,0.368,0.077,Measures earlier click capture before the outcome begins.
2,prior_avg_position,0.091,0.063,Provides the search-position context for attainable CTR.
3,prior_active_days,0.028,0.004,Distinguishes stable history from a thin observation window.
4,prior_clicks,-0.003,0.002,Adds observed click evidence beyond the rounded CTR.


Error by prior-impression quartile:


,prior_impression_quartile,n,mean_actual,mae,under_prediction_pct
0,Q1 lowest,2575,0.157,0.131,37.7
1,Q2,2552,0.233,0.213,38.4
2,Q3,2563,0.429,0.410,38.2
3,Q4 highest,2563,1.015,0.965,26.1


Three largest held-out errors (identifiers intentionally omitted):


,case,prior_impressions,prior_clicks,prior_avg_position,prior_active_days,prior_ctr_pct,future_missed_clicks,model_score,baseline_score,absolute_error,error_direction
63970,case_1,51151.000000,2.000000,7.97,20,0.004,108.48,34.69,150.47,73.79,under-prediction
5251,case_2,639.000000,3.000000,3.19,18,0.469,52.08,0.42,0.00,51.67,under-prediction
5311,case_3,3171.000000,0.000000,3.84,4,0.000,39.80,5.12,0.00,34.68,under-prediction


Wrote reproducible metrics receipt to work/outputs/ml08_model_metrics.json.


## Self-check

- [x] The method matches a continuous ranking target and its complexity is justified
- [x] Feature time precedes outcome time; June remains sealed
- [x] Whole clients are held out with a fixed seed and zero split overlap
- [x] Dummy, Week-4 rule, and model use the same test rows and metrics
- [x] The comparison table includes base rate, median/mean NDCG@20, precision@20, and MAE
- [x] Held-out permutation importance names and explains the top features
- [x] Error analysis shows a difficult slice and three concrete wrong cases
- [x] No identifier, future outcome, trend label, or product flag enters the model
- [x] Claims are observational and decision-support only; limitations are named
- [x] The notebook runs top to bottom without errors and exposes no client names, URLs, private queries, paths, or secrets